### Week 6, Day 2

We're about to create and use our own MCP Server and MCP Client!

It's pretty simple, but it's not super-simple. The excitment around MCP is about how easy it is to share and use other MCP Servers - making our own does involve a bit of work.

Let's review some python code made mostly by a hard-working Engineering Team:

accounts.py

In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown

load_dotenv(override=True)

True

In [2]:
from accounts import Account

In [3]:
account = Account.get("Ed")
account

Account(name='ed', balance=9582.166000000001, strategy='', holdings={'AMZN': 6}, transactions=[3 shares of AMZN at 83.166 each., 3 shares of AMZN at 56.112 each.], portfolio_value_time_series=[('2026-01-13 15:35:30', 9966.502), ('2026-01-13 15:35:34', 9780.502), ('2026-01-13 15:52:57', 10140.166000000001), ('2026-01-13 15:53:00', 9948.166000000001)])

In [4]:
account.buy_shares("AMZN", 3, "Because this bookstore website looks promising")

'Completed. Latest details:\n{"name": "ed", "balance": 9516.034000000001, "strategy": "", "holdings": {"AMZN": 9}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 83.166, "timestamp": "2026-01-13 15:35:30", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 56.112, "timestamp": "2026-01-13 15:52:57", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 22.044, "timestamp": "2026-01-13 16:06:01", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-01-13 15:35:30", 9966.502], ["2026-01-13 15:35:34", 9780.502], ["2026-01-13 15:52:57", 10140.166000000001], ["2026-01-13 15:53:00", 9948.166000000001], ["2026-01-13 16:06:01", 9867.034000000001]], "total_portfolio_value": 9867.034000000001, "total_profit_loss": -132.96600000000035}'

In [5]:
account.report()

'{"name": "ed", "balance": 9516.034000000001, "strategy": "", "holdings": {"AMZN": 9}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 83.166, "timestamp": "2026-01-13 15:35:30", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 56.112, "timestamp": "2026-01-13 15:52:57", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 22.044, "timestamp": "2026-01-13 16:06:01", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-01-13 15:35:30", 9966.502], ["2026-01-13 15:35:34", 9780.502], ["2026-01-13 15:52:57", 10140.166000000001], ["2026-01-13 15:53:00", 9948.166000000001], ["2026-01-13 16:06:01", 9867.034000000001], ["2026-01-13 16:06:01", 9534.034000000001]], "total_portfolio_value": 9534.034000000001, "total_profit_loss": -465.96600000000035}'

In [6]:
account.list_transactions()

[{'symbol': 'AMZN',
  'quantity': 3,
  'price': 83.166,
  'timestamp': '2026-01-13 15:35:30',
  'rationale': 'Because this bookstore website looks promising'},
 {'symbol': 'AMZN',
  'quantity': 3,
  'price': 56.112,
  'timestamp': '2026-01-13 15:52:57',
  'rationale': 'Because this bookstore website looks promising'},
 {'symbol': 'AMZN',
  'quantity': 3,
  'price': 22.044,
  'timestamp': '2026-01-13 16:06:01',
  'rationale': 'Because this bookstore website looks promising'}]

### Now we write an MCP server and use it directly!

In [7]:
# Prüfen Sie, ob fastmcp importiert werden kann
try:
    from mcp.server.fastmcp import FastMCP
    print("✅ fastmcp ist verfügbar!")
except ImportError as e:
    print(f"❌ fastmcp fehlt: {e}")

✅ fastmcp ist verfügbar!


In [8]:
# Now let's use our accounts server as an MCP server

params = {"command": "uv", "args": ["run", "accounts_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()


In [9]:
mcp_tools

[Tool(name='get_balance', title=None, description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'get_balanceOutput', 'type': 'object'}, annotations=None, meta=None),
 Tool(name='get_holdings', title=None, description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, outputSchema={'additionalProperties': {'type': 'integer'}, 'title': 'get_holdingsDictOutput', 'type': 'object'}, annotations=None, meta=None),
 Tool(name='buy_shares', title=None, description=

In [10]:
instructions = "You are able to manage an account for a client, and answer questions about the account."
request = "My name is Ed and my account is under the name Ed. What's my balance and my holdings?"
model = "gpt-4.1-mini"

In [12]:
from openai import AsyncOpenAI
from agents import OpenAIChatCompletionsModel

model = "ministral-3:14b"
ollama_client = AsyncOpenAI(
    base_url="http://localhost:11434/v1", api_key="ollama")

local_model = OpenAIChatCompletionsModel(
    model=model,
    openai_client=ollama_client
)
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="account_manager", instructions=instructions, 
    model=local_model, mcp_servers=[mcp_server])
    with trace("account_manager"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Your current cash balance is **$9,516.03** and your current stock holding is:

- **Amazon (AMZN): 9 shares**

### Now let's build our own MCP Client

In [ ]:
from accounts_client import get_accounts_tools_openai, read_accounts_resource, list_accounts_tools

mcp_tools = await list_accounts_tools()
print(mcp_tools)
openai_tools = await get_accounts_tools_openai()
print(openai_tools)

In [ ]:
request = "My name is Ed and my account is under the name Ed. What's my balance?"

with trace("account_mcp_client"):
    agent = Agent(name="account_manager", instructions=instructions, model=model, tools=openai_tools)
    result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

In [ ]:
context = await read_accounts_resource("ed")
print(context)

In [ ]:
from accounts import Account
Account.get("ed").report()

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercises</h2>
            <span style="color:#ff7800;">Make your own MCP Server! Make a simple function to return the current Date, and expose it as a tool so that an Agent can tell you today's date.<br/>Harder optional exercise: then make an MCP Client, and use a native OpenAI call (without the Agents SDK) to use your tool via your client.
            </span>
        </td>
    </tr>
</table>